# WU PWS — pyramid QC

Reads `dataset/raw/full/outputs/pws_wu_merged_2023-10-29_2026-04-24.nc`,
applies the pyramid QC (sample → hour → day → 30d → station), preserves every
station and every variable, and writes a cleaned `_qc.nc` next to the original.

The QC logic lives in `src/analysis/pws_qc.py` so it's reusable. This notebook
is the **driver**: load → check UTC → run QC → inspect → save → verify. Each
cell's markdown header says what it reads / mutates / outputs.

**Five station outcomes:**
- `as_is`        — per-interval PWS, no transformation applied
- `as_is_asos`   — known NWS airport gauge (KJFK/KLGA/KNYC/…), cumulative-counter detection bypassed
- `differentiated` — looked like a cumulative counter (`mono_frac ≥ 0.97` & `max > 20`), recovered via `diff().clip(lower=0)`
- `dropped`      — failed final total/coverage check; original values preserved in the file for inspection
- `no_rainfall`  — no `rainfall_amount` variable in the group

In [ ]:
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import netCDF4 as nc4
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings('ignore')

REPO_ROOT = Path('../../../').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))
import xarray as _xr
_xr.set_options(file_cache_maxsize=512)
from analysis.netcdf_utils import load_pws_grouped
from analysis.pws_qc import (
    QCConfig, ASOS_STATIONS,
    pyramid_qc, encode_condition, save_qc, read_qc_status,
    verify_utc_time,
)
from analysis.pws_qc import station_map_folium, plot_daily_heatmap

OUT_DIR    = REPO_ROOT / 'dataset' / 'raw' / 'full' / 'outputs'
INPUT_NC   = OUT_DIR / 'pws_wu_merged_2023-10-29_2026-04-24.nc'
OUTPUT_NC  = OUT_DIR / 'pws_wu_merged_2023-10-29_2026-04-24_qc.nc'
LOOKUP_JS  = OUT_DIR / 'pws_wu_merged_2023-10-29_2026-04-24_qc_condition_lookup.json'

for f in [INPUT_NC]:
    print(f"  in : {f.name}  ({f.stat().st_size/1e6:.0f} MB)")
print(f"  out: {OUTPUT_NC.name}  (written by §6)")
print(f"  out: {LOOKUP_JS.name}  (written by §5)")

## §1. Load + UTC sanity check

**Reads:** the merged WU netCDF.<br>
**Outputs:** `pws` (dict of xr.Dataset), variable inventory, confirmation that every group's time axis is UTC.

In [ ]:
pws = load_pws_grouped(INPUT_NC)
print(f'Loaded {len(pws)} stations.\n')

all_vars = sorted({v for ds in pws.values() for v in ds.data_vars})
print(f'Variables seen across stations: {all_vars}\n')

verify_utc_time(pws)

## §2. Station map (real NYC basemap)

**Outputs:** interactive folium map. Pan / zoom / click markers for station IDs.

(The map won't render in static export — open in JupyterLab to interact.)

In [ ]:
station_map_folium({'WU PWS': pws})

## §3. Pre-QC daily heatmap of the worst-looking stations

**Outputs:** heatmap (log10 daily mm + 0.1) of the 25 stations with the highest single-day total. Bands of bright color across whole months = cumulative counters; isolated bright spots = real rain events.

In [ ]:
plot_daily_heatmap(pws, title='WU PWS — daily rainfall (log10 mm+0.1), 25 worst by daily max')
plt.show()

## §4. Run the pyramid QC

**Mutates:** builds `pws_clean` (87 stations preserved) and `qc_log` (per-station record).<br>
**Outputs:** verdict counts + mask totals.

Tune the thresholds via `QCConfig(...)`. Defaults are the ones documented in `src/analysis/pws_qc.py`.

In [ ]:
cfg = QCConfig()  # defaults; override e.g. QCConfig(max_30d_mm=800.0)
pws_clean, qc_log = pyramid_qc(pws, cfg=cfg)

print(f'Total stations: {len(pws_clean)} (input: {len(pws)})')
print('\nStation-level verdict counts:')
print(qc_log['qc_status'].value_counts())
print('\nSamples masked by level:')
for col in ['n_mask_sample', 'n_mask_hour', 'n_mask_day', 'n_mask_30d']:
    print(f'  {col:18s} {qc_log[col].sum():,}')

In [ ]:
# Stations most affected by the pyramid (any_mask = total samples NaN'd)
qc_log['any_mask'] = qc_log[['n_mask_sample', 'n_mask_hour',
                              'n_mask_day', 'n_mask_30d']].sum(axis=1)
qc_log.sort_values('any_mask', ascending=False).head(15)

## §5. Encode `condition` (string) → integer codes + JSON sidecar

**Mutates:** replaces `condition` in each station's Dataset with a float32 array of codes; writes the lookup JSON.

We don't drop the field — it's preserved losslessly via the sidecar. Decode with `lookup = json.load(open(LOOKUP_JS))` and `lookup[str(int(code))]`.

In [ ]:
lookup = encode_condition(pws_clean, LOOKUP_JS, var_name='condition')
print(f'Encoded {len(lookup)} unique condition labels.')
print(f'First 10: {list(lookup.items())[:10]}')
print(f'\nWrote {LOOKUP_JS}')

## §6. Save the cleaned netCDF

**Outputs:** `pws_wu_merged_*_qc.nc` with all 87 stations, full variable set, per-group `qc_status` / `qc_reason` attributes.

Uses `save_grouped_pws` (unmodified) under the hood, then appends the QC attrs via `nc4.Dataset(..., 'a')`.

In [ ]:
save_qc(
    pws_clean, OUTPUT_NC, qc_log,
    extra_global_attrs={'qc_condition_lookup': LOOKUP_JS.name},
)
print(f'Wrote {OUTPUT_NC}  ({OUTPUT_NC.stat().st_size/1e6:.0f} MB, {len(pws_clean)} stations)')

## §7. Re-load + sanity check

**Outputs:** verdict counts as read back from disk, plus a decoded `condition` sample to confirm round-trip.

In [ ]:
df_status = read_qc_status(OUTPUT_NC)
print('qc_status counts in file:')
print(df_status['qc_status'].value_counts())

# Decode a condition sample
lookup_inv = {int(k): v for k, v in json.load(open(LOOKUP_JS)).items()}
pws_verify = load_pws_grouped(OUTPUT_NC, verbose=False)
for sid in pws_verify:
    ds = pws_verify[sid]
    if 'condition' not in ds:
        continue
    codes = [int(c) for c in ds['condition'].values.flat[:30] if not np.isnan(c)][:5]
    if codes:
        print(f'\nDecoded condition sample ({sid}):')
        for c in codes:
            print(f'  code {c:>3d} → {lookup_inv[c]!r}')
        break
df_status.head(10)

## §8. Explore — paste a station ID + event date to see before/after

Edit `EXPLORE_SID` and `EVENT_CENTER` and re-run. Shows the raw vs differentiated rainfall on a ±2-day window around an event.

In [ ]:
EXPLORE_SID  = 'KNYNEWYO1472'
EVENT_CENTER = pd.Timestamp('2024-03-23')
EVENT_PAD    = pd.Timedelta(days=2)

raw = pws[EXPLORE_SID]['rainfall_amount'].squeeze(drop=True).to_series()
raw.index = pd.to_datetime(raw.index)
diff = raw.diff().clip(lower=0)
if len(diff): diff.iloc[0] = 0.0

win0, win1 = EVENT_CENTER - EVENT_PAD, EVENT_CENTER + EVENT_PAD
raw_h  = raw.loc[win0:win1].resample('1h').sum(min_count=1)
diff_h = diff.loc[win0:win1].resample('1h').sum(min_count=1)
raw_c  = raw.loc[win0:win1].fillna(0).cumsum()
diff_c = diff.loc[win0:win1].fillna(0).cumsum()

fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharex=True)
axes[0, 0].plot(raw_h.index, raw_h, marker='.', ms=3, lw=1.0, color='#1f77b4')
axes[0, 0].set_title(f'{EXPLORE_SID} RAW hourly — sum={raw_h.sum():.1f} mm')
axes[0, 1].plot(raw_c.index,  raw_c, lw=1.0, color='#7f7f7f')
axes[0, 1].set_title('RAW cumulative')
axes[1, 0].plot(diff_h.index, diff_h, marker='.', ms=3, lw=1.0, color='#2ca02c')
axes[1, 0].set_title(f'DIFF hourly — sum={diff_h.sum():.1f} mm')
axes[1, 1].plot(diff_c.index, diff_c, lw=1.0, color='#d62728')
axes[1, 1].set_title('DIFF cumulative')
for ax in axes.flat:
    ax.axvspan(EVENT_CENTER, EVENT_CENTER + pd.Timedelta(days=1), color='red', alpha=0.06)
    ax.grid(True, alpha=0.3)
for ax in axes[1]:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %Hh'))
    for lab in ax.get_xticklabels(): lab.set_rotation(30)
plt.tight_layout(); plt.show()

## §9. Per-station before/after grid — full series + event zoom

**Reads:** raw `pws` dict + `qc_log`.<br>
**Outputs:** two grids of 4 stations each:
1. **Full-period** raw vs differentiated, daily + cumulative.
2. **Event zoom** (2024-03-23 ±2 days) for the same 4 stations.

Default picks one station from each QC status — one row per category in a single figure. Edit `EXPLORE_SIDS` to inspect a different set.

In [ ]:
from analysis.pws_qc import plot_per_station_before_after

EVENT = pd.Timestamp('2024-03-23')

def _covers(sid, day):
    t = pd.to_datetime(pws[sid]['time'].values)
    return t.min() <= day <= t.max()

def _pick(status, n=1, must_cover=None):
    sub = qc_log[qc_log['qc_status'] == status]
    if must_cover is not None:
        sub = sub[sub['station'].apply(lambda s: _covers(s, must_cover))]
    if sub.empty: return []
    return list(sub.sort_values('final_total_mm', ascending=False).head(n)['station'])

EXPLORE_SIDS = (_pick('as_is_asos',     1, must_cover=EVENT) +
                _pick('as_is',          1, must_cover=EVENT) +
                _pick('differentiated', 1, must_cover=EVENT) +
                _pick('dropped',        1, must_cover=EVENT))
print(f'Stations covering {EVENT.date()}: {EXPLORE_SIDS}')

# Full-period view
plot_per_station_before_after(pws, EXPLORE_SIDS); plt.show()

# Event zoom (±2 days)
plot_per_station_before_after(
    pws, EXPLORE_SIDS,
    event_day=EVENT, event_pad_days=2,
); plt.show()